# 04 · Presupuesto y desviaciones

Separa la desviación de ingresos y costes para enfocar la revisión mensual en causas accionables.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Actual frente a presupuesto

In [2]:
a=pd.read_csv(RAW/'fact_finance_actual.csv',parse_dates=['month'])
b=pd.read_csv(RAW/'fact_finance_budget.csv',parse_dates=['month'])
df=a.merge(b,on=['month','route_id'],validate='one_to_one')
df['revenue_variance']=df.revenue-df.budget_revenue
df['fuel_variance']=df.budget_fuel_cost-df.fuel_cost
df['other_cost_variance']=(df.budget_variable_cost-df.budget_fuel_cost+df.budget_fixed_allocated)-(df.variable_cost-df.fuel_cost+df.fixed_allocated)
df['ebitda_variance']=df.ebitda-df.budget_ebitda
var=df.groupby('year',as_index=False)[['revenue_variance','fuel_variance','other_cost_variance','ebitda_variance']].sum()
var.to_csv(TABLES/'04_budget_variance_annual.csv',index=False)
display(var)

   year  revenue_variance  fuel_variance  other_cost_variance  ebitda_variance
0  2024     -2,508,331.30  -1,257,033.76        -1,502,152.05    -5,267,517.11
1  2025     -4,096,310.79  -1,526,576.19        -1,399,979.46    -7,022,866.44
2  2026     -3,350,591.59  -1,052,007.40        -1,407,291.83    -5,809,890.82


## Desviación por ruta

In [3]:
by_route=df.groupby('route_id',as_index=False).agg(actual_ebitda=('ebitda','sum'),budget_ebitda=('budget_ebitda','sum'),ebitda_variance=('ebitda_variance','sum'))
by_route['variance_pct']=by_route.ebitda_variance/by_route.budget_ebitda.abs()
by_route.to_csv(TABLES/'04_budget_variance_route.csv',index=False)
display(by_route.sort_values('ebitda_variance'))

  route_id  actual_ebitda  budget_ebitda  ebitda_variance  variance_pct
0  BCN-IBZ   2,946,777.58   6,234,336.09    -3,287,558.51         -0.53
5  VAL-IBZ   4,209,438.72   7,397,607.10    -3,188,168.38         -0.43
1  BCN-PMI   3,945,481.57   6,911,893.00    -2,966,411.43         -0.43
4  DEN-PMI  14,432,338.83  16,940,250.87    -2,507,912.04         -0.15
6  VAL-PMI   4,139,280.54   6,569,151.17    -2,429,870.63         -0.37
3  DEN-IBZ   4,039,550.90   6,063,619.72    -2,024,068.82         -0.33
2  DEN-FOR   5,871,819.54   7,568,104.10    -1,696,284.56         -0.22


## Puente de desviación

In [4]:
bridge=df[['revenue_variance','fuel_variance','other_cost_variance']].sum().rename({'revenue_variance':'Ingresos','fuel_variance':'Combustible','other_cost_variance':'Otros costes'})
colors=['#2dd4bf' if x>=0 else '#ef4444' for x in bridge]
plt.figure(figsize=(9,5)); plt.bar(bridge.index,bridge.values/1e6,color=colors); plt.axhline(0,color='#0f172a',lw=.8); plt.ylabel('Impacto M€'); plt.title('Puente acumulado de desviación EBITDA vs presupuesto')
plt.tight_layout(); plt.savefig(FIGURES/'04_puente_desviacion.png',dpi=180,bbox_inches='tight'); plt.show()

## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.